# Notebook 17.1  A small fairness-and-security audit for an Arabic speech system

**Goal.** Quantify per-dialect and per-gender disparities for a recognizer, run a simple spoof/deepfake detector and measure how it drops on an unseen generator, then write a short model-card-style summary.

**What runs here.** Everything runs with no downloads. We evaluate one model across several dialects and both genders on matched synthetic data, computing per-group error and the disparity gap rather than a pooled average; we train and test a simple spoof detector on ArFake-style features and report accuracy on a seen generator and the drop on an unseen one; and we emit a model card. A synthetic fallback makes the notebook self-contained, and the markdown marks where real recordings and a real detector plug in.

**The point.** Trust is measured, not asserted. A single average can hide the dialect or gender a system fails, and a detector that looks strong on familiar fakes can collapse on new ones. Reporting per group, and on unseen attacks, is the honest practice of Chapter 17.

## 1. Setup

In [ ]:
import numpy as np
from collections import defaultdict
rng = np.random.default_rng(17)   # reproducible
print('ready')

## 2. A matched, multi-dialect, two-gender evaluation set

To audit fairness we need *matched* data: the same content spoken across dialects and both genders, so that an error difference reflects the system, not the material. Here we synthesize per-group error rates with a deliberate gap (one dialect, and the female group within it, served worse). Replace this block with a real model evaluated on matched recorded speech, keeping the per-group structure.

In [ ]:
DIALECTS = ['MSA', 'Gulf', 'Egyptian', 'Maghrebi']
GENDERS  = ['female', 'male']
N_PER_GROUP = 200

# 'true' per-group error tendencies (illustrative): Maghrebi worst, with a gender gap
base = {'MSA':0.08, 'Gulf':0.15, 'Egyptian':0.17, 'Maghrebi':0.30}
gender_gap = {'MSA':0.01, 'Gulf':0.03, 'Egyptian':0.03, 'Maghrebi':0.10}  # female penalty

def simulate_errors(dialect, gender):
    p = base[dialect] + (gender_gap[dialect] if gender == 'female' else 0.0)
    # each utterance is correct/incorrect; WER approximated by per-utterance error rate
    return rng.random(N_PER_GROUP) < p   # True = error

errors = {(d, g): simulate_errors(d, g) for d in DIALECTS for g in GENDERS}
print('groups:', len(errors), '| utterances per group:', N_PER_GROUP)

## 3. Per-group error and the disparity gap

We report the error rate for every (dialect, gender) cell, the pooled average, and the disparity gap (worst minus best group). The gap, not the average, is the fairness headline.

In [ ]:
rates = {k: float(v.mean()) for k, v in errors.items()}

print('per-group error rate:')
print('%-10s %-9s %-9s' % ('dialect', 'female', 'male'))
for d in DIALECTS:
    print('%-10s %-9.3f %-9.3f' % (d, rates[(d,'female')], rates[(d,'male')]))

pooled = float(np.mean(list(rates.values())))
worst_k = max(rates, key=rates.get)
best_k  = min(rates, key=rates.get)
gap = rates[worst_k] - rates[best_k]
print('\npooled average error: %.3f' % pooled)
print('best group : %-22s %.3f' % (str(best_k), rates[best_k]))
print('worst group: %-22s %.3f' % (str(worst_k), rates[worst_k]))
print('DISPARITY GAP (worst - best): %.3f' % gap)
FAIRNESS_THRESHOLD = 0.15
print('exceeds fairness threshold %.2f? %s' % (FAIRNESS_THRESHOLD, gap > FAIRNESS_THRESHOLD))

The pooled average looks acceptable, but the gap reveals that one group (Maghrebi female) is failed far more than the best group (MSA male). This is exactly the harm an average hides, and why Chapter 17 insists on disaggregated reporting.

## 4. A spoof / deepfake detector and the unseen-generator drop

Security has the same honesty problem. A detector trained on fakes from one text-to-speech generator may not catch fakes from another. We build a tiny detector on synthetic 'embedding' features for genuine vs spoofed speech, then test it on a *seen* generator and an *unseen* one. Replace the features with ArFake-style embeddings and the classifier with a real anti-spoofing model; the seen-vs-unseen protocol stays the same.

In [ ]:
D = 16  # feature dim
def make_samples(n, center, scale=1.0):
    return rng.normal(center, scale, size=(n, D))

# genuine speech cluster
genuine_center = np.zeros(D)
# generator A (seen in training): a distinct artifact direction
genA_center = np.full(D, 0.0); genA_center[:4] = 2.2
# generator B (unseen): a DIFFERENT artifact direction
genB_center = np.full(D, 0.0); genB_center[8:12] = 2.2

# training set: genuine + spoofed-by-A only
Xg = make_samples(600, genuine_center)
Xa = make_samples(600, genA_center)
Xtr = np.vstack([Xg, Xa]); ytr = np.array([0]*600 + [1]*600)  # 0 genuine, 1 spoof

# nearest-centroid detector (transparent baseline)
c_gen = Xtr[ytr==0].mean(0); c_spoof = Xtr[ytr==1].mean(0)
def detect(X):
    dg = ((X - c_gen)**2).sum(1); ds = ((X - c_spoof)**2).sum(1)
    return (ds < dg).astype(int)   # 1 = spoof

def eval_detector(gen_center, label):
    Xg_te = make_samples(300, genuine_center); Xs_te = make_samples(300, gen_center)
    X = np.vstack([Xg_te, Xs_te]); y = np.array([0]*300 + [1]*300)
    acc = float((detect(X) == y).mean())
    print('%-28s detection accuracy = %.3f' % (label, acc))
    return acc

acc_seen   = eval_detector(genA_center, 'seen generator (A)')
acc_unseen = eval_detector(genB_center, 'unseen generator (B)')
print('\nDROP on unseen generator: %.3f -> %.3f  (%.3f)' % (acc_seen, acc_unseen, acc_seen - acc_unseen))

The detector is strong on the generator it was trained against and much weaker on the unseen one. Reporting only the seen-generator number would badly overstate real-world robustness, which is why the unseen-generator drop is the honest metric for deepfake detection.

## 5. A model-card-style summary

Finally, we emit a short model card: what was tested, the per-group results, the disparity gap, the spoof-detection results with the unseen-generator drop, and the data provenance. This is the documentation artifact Chapter 17 argues should accompany any deployed system.

In [ ]:
lines = []
lines.append('MODEL CARD (illustrative)')
lines.append('='*40)
lines.append('Task: Arabic ASR + spoof detection audit')
lines.append('Data provenance: SYNTHETIC (teaching fallback) -- replace with recorded, human-labeled speech')
lines.append('')
lines.append('Fairness (per-group error rate):')
for d in DIALECTS:
    lines.append('  %-10s female %.3f | male %.3f' % (d, rates[(d,'female')], rates[(d,'male')]))
lines.append('  pooled %.3f | disparity gap %.3f | threshold %.2f -> %s'
             % (pooled, gap, FAIRNESS_THRESHOLD, 'FAIL' if gap>FAIRNESS_THRESHOLD else 'pass'))
lines.append('')
lines.append('Security (spoof detection):')
lines.append('  seen generator   acc %.3f' % acc_seen)
lines.append('  unseen generator acc %.3f  (drop %.3f)' % (acc_unseen, acc_seen-acc_unseen))
lines.append('')
lines.append('Recommended mitigation: collect Maghrebi (esp. female) data and retrain;')
lines.append('  add fakes from diverse generators; re-audit per group and on unseen generators.')
print('\n'.join(lines))

## 6. What this shows, and a mitigation plan

The audit surfaces two concrete harms that a single average would hide: a fairness gap that falls hardest on Maghrebi female speakers, and a deepfake detector that does not generalize to an unseen generator. The proposed mitigation is specific and testable: collect more data for the worst-served group and retrain, broaden the spoof-training generators, and then re-run this exact audit, reporting per group and on unseen attacks, to confirm the gap and the drop have narrowed.

To make this real, replace Section 2 with a deployed recognizer evaluated on matched, recorded, human-labeled Arabic speech across dialects and genders, and Section 4 with ArFake-style genuine and spoofed embeddings from several generators, holding one out as unseen. The audit logic, per-group error, disparity gap, seen-versus-unseen detection, and the model card, stays exactly as written; only the data and model are swapped in.